# Proyecto 2: PageRank sobre redes reales
**Curso:** IMT2230 — Álgebra Lineal Avanzada y Modelamiento, Pontificia Universidad Católica de Chile

**Red elegida:** *Political blogs* (`moreno_blogs` en KONECT) — hipervínculos entre blogs políticos de EE.UU. durante la elección presidencial de 2004.

**Integrantes:** Alonso Aqueveque, Alex Nieves, Francisco Reyes

**Repositorio GitHub:** https://github.com/Prodigy0101/Proyecto-2-Algebra-Lineal-Avanzada-y-Modelamiento

---
## Cómo ejecutar este notebook

1. Requiere **Python ≥ 3.9** con las librerías: `numpy`, `scipy`, `networkx`, `matplotlib`, `pandas`.
2. Ejecutar las celdas **en orden**. Funciona igual en Google Colab o en Jupyter local.
3. La primera celda de código **descarga automáticamente** el dataset (≈1 MB). Si ya existe `data/polblogs.gml`, no lo vuelve a descargar.
4. Las figuras se guardan en la carpeta `figs/` (se usan en el informe PDF).

> **Fuente de los datos:** la red corresponde a `moreno_blogs` del catálogo KONECT (http://konect.cc/networks/moreno_blogs/), cuyo origen es Adamic, L. & Glance, N. (2005), *"The political blogosphere and the 2004 U.S. election: divided they blog"*. Aquí descargamos el archivo GML original de Adamic & Glance (idéntico contenido, pero incluye además la **URL de cada blog** y su **orientación política**, atributos que KONECT no distribuye en su edge-list y que necesitamos para P7).

## Importaciones y extracción de documento

In [ ]:

import os, urllib.request
import numpy as np
import scipy.sparse as sp
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
# para comparar rankings en P6
from scipy.stats import rankdata

plt.rcParams['figure.dpi'] = 110
os.makedirs('data', exist_ok=True)
os.makedirs('figs', exist_ok=True)

#Descarga del dataset
# URL GitHub espejo del GML original de Adamic & Glance (2005).
URL = "https://raw.githubusercontent.com/Dammy-Olude/polblogs_analysis/master/Datasets/polblogs.gml"
PATH = "data/polblogs.gml"

if not os.path.exists(PATH):
    print("Descargando dataset...")
    urllib.request.urlretrieve(URL, PATH)
print("Dataset disponible en:", PATH, f"({os.path.getsize(PATH)/1e6:.2f} MB)")

---
## P1 — Elección y descripción de la red

**Nota de preprocesamiento:** el archivo original registra 1 490 nodos y 19 090 enlaces, pero incluye 266 blogs sin ningún enlace (nodos aislados), algunos enlaces duplicados y 3 auto-enlaces. Igual que hace KONECT, trabajamos con el grafo simple (sin duplicados ni self-loops) y sin nodos aislados, quedando $n=1\,224$ y $m=19\,022$.

In [ ]:
#Carga del grafo
#leemos como MultiDiGraph por las aristas duplicadas
gml_text = open(PATH, encoding='utf8').read().replace('graph [', 'graph [\n  multigraph 1', 1)
MG = nx.parse_gml(gml_text.splitlines(), label='id')
print(f"Grafo crudo: {MG.number_of_nodes()} nodos, {MG.number_of_edges()} enlaces (con duplicados)")

# Convertimos a grafo dirigido SIMPLE
G = nx.DiGraph(MG)
G.remove_edges_from(list(nx.selfloop_edges(G)))        # self-loops
G.remove_nodes_from([v for v in G.nodes if G.degree(v) == 0])   # nodos aislados

n = G.number_of_nodes()
m = G.number_of_edges()
labels = nx.get_node_attributes(G, 'label')   # labels[x]: URL del blog x
values = nx.get_node_attributes(G, 'value')   # values[y]: 0 = liberal, 1 = conservador
print(f"Grafo final tiene {n} nodos y {m} aristas")
print(f"Con {sum(1 for x in values.values() if x==0)} blogs liberales y "
      f"{sum(1 for x in values.values() if x==1)} blogs conservadores")

In [ ]:
din  = dict(G.in_degree())    # din[v]:  cantidad de entrada al nodo v
dout = dict(G.out_degree())   # dout[v]: cantidad de salida del nodo v

dangling = [v for v in G.nodes if dout[v] == 0]        # nodos sin salida
v_max_in = max(G.nodes, key=lambda v: din[v])          # nodo con la mayor cantidad de entrada

stats = pd.DataFrame({
    'Estadística': ['Número de nodos  n', 'Número de aristas  m',
                    'Grado de entrada medio  d̄_in', 'Grado de salida medio  d̄_out',
                    'Nodo de mayor grado de entrada', 'Densidad  m/(n(n−1))',
                    'Nodos colgantes (out-degree 0)'],
    'Valor': [n, m, f"{m/n:.3f}", f"{m/n:.3f}",
              f"{labels[v_max_in]}  (d_in = {din[v_max_in]})",
              f"{m/(n*(n-1)):.5f}",
              f"{len(dangling)}  ({100*len(dangling)/n:.1f}% del total)"]
})
stats

**Comentarios a la tabla.** El grado medio de entrada y de salida coinciden siempre ($\bar d^{in}=\bar d^{out}=m/n\approx 15.5$) porque cada arista aporta exactamente una unidad a cada suma. La densidad ($\approx 1.3\%$) confirma que la red es **dispersa**, lo que justifica usar `scipy.sparse`. El nodo más enlazado es **dailykos.com** (337 enlaces entrantes), el mayor blog demócrata de la época. Hay **160 nodos colgantes** ($13.1\%$), un porcentaje relevante que habrá que reparar al construir $S$ (P4).

---
## P2 — Pregunta e hipótesis inicial

En el informe, en el tercer punto, se presenta la pregunta y la hipotesis.

---
## P3 — Análisis exploratorio de la red

### (a) Distribución del grado de entrada y de salida

In [ ]:
din_arr  = np.array([din[v]  for v in G.nodes])   # vector de cantidades de entrada
dout_arr = np.array([dout[v] for v in G.nodes])   # vector de cantidades de salida

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, arr, name, c in [(axes[0], din_arr, 'entrada', 'tab:blue'),
                         (axes[1], dout_arr, 'salida', 'tab:orange')]:
    # Contamos cuántos nodos tienen cada cantidad k y lo graficamos en escala log-log:
    # las redes reales suelen mostrar colas pesadas (pocos nodos con cantidad enorme).
    ks, cnt = np.unique(arr, return_counts=True)
    ax.scatter(ks + 1, cnt, s=14, color=c)         # +1 para poder mostrar cantidad 0 en escala log
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel(f'cantidad de {name} + 1'); ax.set_ylabel('número de nodos')
    ax.set_title(f'Distribución de la cantidad de {name}')
    ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig('figs/fig_dist_grados.png', bbox_inches='tight'); plt.show()

print(f"Cantidad de entrada: mediana={np.median(din_arr):.0f}, máximo={din_arr.max()}")
print(f"Cantidad de salida : mediana={np.median(dout_arr):.0f}, máximo={dout_arr.max()}")

**Lectura del gráfico.** Ambas distribuciones tienen **cola pesada** (patrón aproximadamente lineal en escala log-log): la mayoría de los blogs tiene pocos enlaces y unos pocos concentran cientos. Esta heterogeneidad es el escenario donde un ranking como PageRank es informativo.

### (b) Resolver los nodos colgantes

In [ ]:
print(f"Nodos colgantes (cantidad de salida 0): {len(dangling)} de {n}  ({100*len(dangling)/n:.2f}%)")
print("Ejemplos:", [labels[v] for v in dangling[:5]])

**¿Por qué son problemáticos?** En la matriz de hipervínculos $H$, la columna de un nodo colgante $j$ es cero (no reparte probabilidad a nadie). Entonces $H$ no es columna-estocástica: $\mathbf{1}^T H \mathbf{r} < 1$ cuando $r_j>0$, es decir, en cada iteración se pierde masa de probabilidad por los colgantes y el vector deja de ser una distribución. En términos de la marcha aleatoria, el caminante llega a un blog sin enlaces de salida y queda atrapado sin regla para continuar.

Para resolver esto se reemplaza cada columna cero por el vector uniforme $\mathbf{1}/n$, construyendo la matriz columna-estocástica
$$S = H + \tfrac{1}{n}\,\mathbf{a}\mathbf{1}^T,\qquad a_j = \begin{cases}1 & \text{si $j$ es colgante}\\ 0 & \text{si no}\end{cases}$$
Así al llegar a un blog sin salidas, el caminante se mueve a cualquier blog de la red con probabilidad uniforme. Con esto toda columna de $S$ suma 1 y la iteración preserva $\|\mathbf r\|_1=1$. Lo implementamos en P4.

### (c) Top-10 por grado de entrada y por grado de salida

In [ ]:
#P3(c): tablas top-10
pol = lambda v: 'liberal' if values[v] == 0 else 'conservador'   # decodifica el atributo 'value'

top10_in  = sorted(G.nodes, key=lambda v: -din[v])[:10]
top10_out = sorted(G.nodes, key=lambda v: -dout[v])[:10]

tabla_in = pd.DataFrame([(labels[v], din[v], dout[v], pol(v)) for v in top10_in],
                        columns=['Blog', 'd_in', 'd_out', 'Orientación'],
                        index=range(1, 11))
tabla_out = pd.DataFrame([(labels[v], dout[v], din[v], pol(v)) for v in top10_out],
                         columns=['Blog', 'd_out', 'd_in', 'Orientación'],
                         index=range(1, 11))
print("=== Top-10 por grado de ENTRADA ==="); display(tabla_in)
print("=== Top-10 por grado de SALIDA ==="); display(tabla_out)

**Observación.** El top de entrada se mantiene repartido parejamente entre conservadores y liberales. El top de salida es distinto: lo encabezan blogs-directorio que enlazan a muchos otros (p. ej. blogsforbush.com, que además es muy enlazado). Que ambos rankings difieran ya sugiere que entrada y salida capturan roles distintos: autoridad vs. hub.

### (d) ¿Es la red fuertemente conexa?

In [ ]:
#P3(d): conectividad
print("¿Fuertemente conexa?:", nx.is_strongly_connected(G))
sccs = sorted(nx.strongly_connected_components(G), key=len, reverse=True)
wccs = sorted(nx.weakly_connected_components(G), key=len, reverse=True)
print(f"Componentes fuertemente conexas: {len(sccs)}; la mayor tiene {len(sccs[0])} nodos "
      f"({100*len(sccs[0])/n:.1f}% de la red)")
print(f"Componentes débilmente conexas : {len(wccs)}; tamaños = {[len(c) for c in wccs]}")

La red NO es fuertemente conexa: tiene 422 componentes fuertes y la mayor concentra solo 793 nodos (64.8%); además existen 2 componentes débiles (una pareja de blogs aislada del resto). Sobre el grafo "crudo" la marcha aleatoria no tendría una distribución estacionaria única. Sin embargo, esto no impide aplicar PageRank: la Matriz de Google $G = \alpha S + \frac{1-\alpha}{n}\mathbf{1}\mathbf{1}^T$ tiene todas sus entradas positivas ($G_{ij} \ge \frac{1-\alpha}{n} > 0$), por lo que es irreducible y aperiódica, o sea ergódica, independientemente de la estructura del grafo subyacente, y el Teorema de Perron–Frobenius garantiza una única distribución estacionaria positiva.

---
## P4 — Construcción de la Matriz de Google

### (a) Matriz de hipervínculos $H$

$$H_{ij} = \begin{cases} \dfrac{1}{\mathrm{out}(j)} & \text{si } j \to i \\[4pt] 0 & \text{si no}\end{cases}$$

Cada columna $j$ reparte la probabilidad uniformemente entre los enlaces de salida del nodo $j$. Si $j$ no es colgante, su columna tiene $\mathrm{out}(j)$ entradas iguales a $1/\mathrm{out}(j)$, de modo que suma $\mathrm{out}(j)\cdot\frac{1}{\mathrm{out}(j)} = 1$. Las columnas cero se identifican simplemente sumando cada columna y detectando las que suman 0; deben coincidir con los nodos colgantes contados en P3(b): 160 columnas cero, el 13.07% del total.

In [ ]:
#P4(a): matriz de hipervínculos H
nodes = list(G.nodes)
idx = {v: i for i, v in enumerate(nodes)}    # idx[v]: índice de columna/fila del nodo v

rows, cols, vals = [], [], []                # tripletas (i, j, valor) para la matriz dispersa
for j in nodes:
    if dout[j] > 0:
        w = 1.0 / dout[j]                    # peso 1/out(j) para cada enlace que sale de j
        for _, i_dest in G.out_edges(j):     # j -> i_dest
            rows.append(idx[i_dest]); cols.append(idx[j]); vals.append(w)

H = sp.csr_matrix((vals, (rows, cols)), shape=(n, n))   # H dispersa

#columnas que suman 1 y columnas cero
col_sums = np.asarray(H.sum(axis=0)).ravel()     # suma de cada columna de H
a = (col_sums == 0).astype(float)                # a[j] = 1 si la columna j es cero (nodo colgante)
print(f"Entradas no nulas de H: {H.nnz}")
print(f"Columnas cero de H: {int(a.sum())}  ({100*a.sum()/n:.2f}% del total) ")
print(f"Columnas no colgantes suman 1: {np.allclose(col_sums[a == 0], 1.0)}")

Notamos que las columnas cero de H coinciden con los nodos colgantes.

### (b) Matriz columna-estocástica $S$

$$S = H + \tfrac{1}{n}\,\mathbf{a}\mathbf{1}^T$$

donde $a_j=1$ si el nodo $j$ es colgante. Esto reemplaza cada columna cero por el vector uniforme $\mathbf 1/n$.

Notar $\mathbf{a}\mathbf{1}^T$ es una matriz con 160 columnas densas; materializarla arruinaría la dispersión. Por eso no formamos $S$ explícitamente sinó que en la iteración usaremos $S\mathbf r = H\mathbf r + \frac{(\mathbf a^T\mathbf r)}{n}\,\mathbf 1$. Aun así, aquí verificamos numéricamente que todas las columnas de $S$ suman 1.

In [ ]:

col_sums_S = col_sums + a
print("Todas las columnas de S suman 1:", np.allclose(col_sums_S, 1.0))
print("mín =", col_sums_S.min(), " máx =", col_sums_S.max())

Estos valores se procuden al computar numeros periodicos, pero su proximidad al 1 nos dice que si se cumple la condición.

### (c) Matriz de Google $G$

$$G = \alpha S + \frac{1-\alpha}{n}\,\mathbf{1}\mathbf{1}^T$$

**Elección de $\alpha$: usamos $\alpha = 0.85$**, el valor clásico propuesto por Brin & Page. La justificación es que con $\alpha$ cercano a 1 el ranking es más fiel a la estructura real de enlaces, pero la convergencia de la iteración de potencias se hace lenta (la tasa de convergencia es $\approx \alpha$, pues $|\lambda_2(G)| \le \alpha$) y el resultado se vuelve más sensible a perturbaciones. Por contraparte si $\alpha$ es pequeño entoces converge rápido pero el ranking se acerca al uniforme y pierde información. Con $\alpha=0.85$ se espera reducir el error en un factor ~$0.85$ por iteración (unas 100–150 iteraciones para $\varepsilon = 10^{-10}$), lo que verificaremos empíricamente en P5(b).

**¿Por qué $G > 0$?** Para todo par $(i,j)$: $G_{ij} = \alpha S_{ij} + \frac{1-\alpha}{n} \ge \frac{1-\alpha}{n} > 0$, ya que $S_{ij}\ge 0$ y $\alpha<1$. El término de teleportación $\frac{1-\alpha}{n}\mathbf 1\mathbf 1^T$ aporta una probabilidad mínima estrictamente positiva de saltar de cualquier nodo a cualquier otro; por eso $G$ es una matriz positiva (y columna-estocástica), luego ergódica.

Igual que con $S$, **no construimos $G$ densa** ($1224^2 \approx 1.5$ millones de entradas todas positivas): usamos
$$G\mathbf r = \alpha H\mathbf r + \alpha\,\frac{\mathbf a^T \mathbf r}{n}\,\mathbf 1 + \frac{1-\alpha}{n}\,\mathbf 1 \qquad (\text{usando } \|\mathbf r\|_1 = 1),$$
que solo requiere un producto disperso $H\mathbf r$ ($O(m)$ operaciones) por iteración.

In [ ]:
#P4(c): producto matriz-vector G@r sin construir G
alpha = 0.85

def google_matvec(r, H=H, a=a, alpha=alpha, n=n):
    '''Calcula G @ r de forma eficiente:  alpha*H@r  +  alpha*(a·r)/n * 1  +  (1-alpha)/n * 1.
    Supone ||r||_1 = 1 (r es una distribución de probabilidad).'''
    return alpha * (H @ r) + (alpha * (a @ r) + (1 - alpha)) / n

# Chequeo: G@r debe seguir siendo distribución (suma 1) para cualquier distribución r
r_test = np.random.rand(n); r_test /= r_test.sum()
print("||G r||_1 =", google_matvec(r_test).sum(), " (debe ser 1)")

---
## P5 — Cálculo del PageRank mediante iteración de potencias

### (a) Iteración desde $\mathbf r^{(0)} = \mathbf 1/n$ hasta $\|\mathbf r^{(k+1)}-\mathbf r^{(k)}\|_1 < 10^{-10}$

In [ ]:
# iteración de potencias
def power_iteration(matvec, n, tol=1e-10, max_iter=1000):
    '''Iteración de potencias r^(k+1) = G r^(k) partiendo del vector uniforme.
    Devuelve: r (aproximación del PageRank), lista de iterados y lista de errores ||r^(k+1)-r^(k)||_1.'''
    r = np.ones(n) / n           # r^(0): distribución uniforme
    iterates, errors = [r.copy()], []
    for k in range(max_iter):
        r_new = matvec(r)                      # r^(k+1) = G r^(k)
        err = np.abs(r_new - r).sum()          # error en norma 1 entre iterados consecutivos
        errors.append(err)
        r = r_new
        iterates.append(r.copy())
        if err < tol:
            break
    return r, iterates, errors

r_star, iterates, errors = power_iteration(google_matvec, n)
n_iters = len(errors)
print(f"Convergió en {n_iters} iteraciones (tolerancia 1e-10, norma 1)")

La iteración converge en 107 iteraciones, dentro de la cota teórica $\alpha^k \approx 10^{-10} \Rightarrow k \approx \frac{-10\ln 10}{\ln 0.85} \approx 142$ (converge un poco antes porque $|\lambda_2|$ puede ser menor que $\alpha$ y el error inicial no es 1).

### (b) Curva de convergencia

In [ ]:
#P5(b): curva de convergencia en escala logarítmica
dist_to_star = [np.abs(rk - r_star).sum() for rk in iterates[:-1]]   # distancia de cada iterado al límite

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.semilogy(range(len(dist_to_star)), dist_to_star, 'o-', ms=3, label=r'$\|\mathbf{r}^{(k)}-\mathbf{r}^*\|_1$')
# referencia teórica: decaimiento geométrico con razón alpha
ref = dist_to_star[0] * alpha ** np.arange(len(dist_to_star))
ax.semilogy(range(len(dist_to_star)), ref, '--', color='gray', label=r'referencia $C\,\alpha^k$ ($\alpha=0.85$)')
ax.set_xlabel('iteración  k'); ax.set_ylabel('error (norma 1, escala log)')
ax.set_title('Convergencia de la iteración de potencias')
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig('figs/fig_convergencia.png', bbox_inches='tight'); plt.show()

# cociente entre errores consecutivos para verificar la razón geométrica
ratios = np.array(errors[1:]) / np.array(errors[:-1])
print(f"Razón media de decaimiento (últimas 20 iteraciones): {ratios[-20:].mean():.4f}  (alpha = {alpha})")

En escala logarítmica la curva es una recta, es decir el decaimiento es geométrico. La razón empírica entre errores consecutivos es ≈ 0.8498 ≈ α = 0.85, tal como predice la teoría ($|\lambda_2(G)|\le\alpha$).

### (c) Verificación de $\|\mathbf r^*\|_1 = 1$ y $r_i^* > 0$

In [ ]:
#P5(c): propiedades del vector PageRank
print(f"||r*||_1 = {r_star.sum():.12f}")
print(f"mín r_i  = {r_star.min():.3e}   (cota teórica (1-alpha)/n = {(1-alpha)/n:.3e})")
print(f"¿Todas las entradas positivas?: {(r_star > 0).all()}")

Ambas propiedades están garantizadas por la teoría:

- $\|\mathbf r^*\|_1=1$: $G$ es columna-estocástica, así que $\mathbf 1^T G = \mathbf 1^T$ y cada iteración preserva la suma del vector. Partiendo de $\mathbf 1/n$ (que suma 1), el límite también suma 1.
- $r_i^*>0$: como toda entrada de $G$ es $\ge \frac{1-\alpha}{n}$, se tiene $r_i^* = (G\mathbf r^*)_i \ge \frac{1-\alpha}{n} > 0$ (es lo que asegura Perron–Frobenius para matrices positivas). El mínimo observado respeta la cota.

### (d) Tabla de los 20 nodos de mayor PageRank

In [ ]:
#P5(d): top-20 PageRank
orden = np.argsort(-r_star)     # índices ordenados de mayor a menor PageRank

top20 = pd.DataFrame(
    [(rank, labels[nodes[i]], f"{r_star[i]:.6f}", din[nodes[i]], dout[nodes[i]], pol(nodes[i]))
     for rank, i in enumerate(orden[:20], start=1)],
    columns=['Rango', 'Blog (identificador)', 'PageRank r*_i', 'd_in', 'd_out', 'Orientación']
).set_index('Rango')
top20

---
## P6 — PageRank vs. grado de entrada

### (a) Dispersión y correlación de Pearson

In [ ]:
#P6(a): dispersión cantidad de entrada vs PageRank + correlación
pearson = np.corrcoef(din_arr_o := np.array([din[v] for v in nodes]), r_star)[0, 1]

fig, ax = plt.subplots(figsize=(7.5, 5.5))
colores = ['tab:blue' if values[v] == 0 else 'tab:red' for v in nodes]
ax.scatter(din_arr_o, r_star, s=12, alpha=.45, c=colores)

# destacamos los 10 nodos de mayor PageRank con su nombre
for i in orden[:10]:
    ax.scatter(din_arr_o[i], r_star[i], s=70, edgecolor='k',
               c='tab:blue' if values[nodes[i]] == 0 else 'tab:red', zorder=3)
    ax.annotate(labels[nodes[i]], (din_arr_o[i], r_star[i]), fontsize=7,
                xytext=(4, 4), textcoords='offset points')

ax.set_xlabel('cantidad de entrada  d_in'); ax.set_ylabel('PageRank  r*')
ax.set_title(f'PageRank vs. cantidad de entrada  (Pearson = {pearson:.3f})')
ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig('figs/fig_scatter_pr_vs_in.png', bbox_inches='tight'); plt.show()
print(f"Coeficiente de correlación de Pearson: {pearson:.4f}")

La correlación es muy alta (Pearson ≈ 0.957): en esta red, ser muy enlazado casi siempre implica alto PageRank, aunque la nube se abre para valores intermedios. La interpretación de este resultado está en el informe, en el sexto punto.

### (b) Nodos donde PageRank e in-degree difieren significativamente

In [ ]:
#P6(b): nodos discrepantes comparando rankings
rank_pr = rankdata(-r_star, method='min')     # posición de cada nodo en el ranking por PageRank
rank_in = rankdata(-din_arr_o, method='min')  # posición de cada nodo en el ranking por cantidad de entrada

def ficha(nombre):
    '''Imprime la ficha comparativa de un blog dado su nombre.'''
    v = [u for u in nodes if labels[u] == nombre][0]; i = idx[v]
    print(f"{nombre:22s} d_in={din[v]:4d} (rank #{int(rank_in[i]):3d})  ->  "
          f"PageRank={r_star[i]:.6f} (rank #{int(rank_pr[i]):3d})   d_out={dout[v]}")

print("— Nodo con PageRank ALTO pero in-degree moderado:")
ficha('freerepublic.com'); ficha('vodkapundit.com')
print("\n— Nodo con in-degree ALTO pero PageRank relativamente bajo:")
ficha('redstate.org'); ficha('lt-smash.us')

# miramos el PageRank de los vecinos entrantes de freerepublic.com
vecinos_fr = [u for u, _ in G.in_edges([v for v in nodes if labels[v] == 'freerepublic.com'][0])]
pr_vecinos = sorted([(r_star[idx[u]], labels[u]) for u in vecinos_fr], reverse=True)[:5]
print("\nTop-5 blogs que enlazan a freerepublic.com (por su propio PageRank):")
for prv, lab in pr_vecinos: print(f"   {lab:35s} r = {prv:.6f}")

Encontramos los casos discrepantes: freerepublic.com y vodkapundit.com suben en PageRank respecto a su cantidad de entrada (#50 → #20 y #31 → #13), porque los enlazan blogs muy influyentes, mientras que redstate.org y lt-smash.us caen (#80 → #164 y #94 → #181), porque sus enlaces vienen de blogs periféricos. La interpretación de estas discrepancias se desarrolla en el informe, en el sexto punto.

---
## P7 — Interpretación de los resultados

### (a) ¿Tienen sentido los nodos de mayor PageRank?

Sí: el top-20 coincide con los blogs reconocidos de la blogosfera política de 2004 (dailykos.com, atrios.blogspot.com, instapundit.com, drudgereport.com, etc.), lo que valida el método usando solo la estructura de enlaces. El detalle de cada blog y su contexto histórico está en el informe, en el quinto punto.

### (b) Subgrafo inducido por los 40 nodos de mayor PageRank

In [ ]:
#P7(b): subgrafo inducido por el top-40 de PageRank
K = 40
top_nodes = [nodes[i] for i in orden[:K]]           # los K nodos de mayor PageRank
SG = G.subgraph(top_nodes)                          # subgrafo inducido (con todas las aristas entre ellos)
pr_map = {nodes[i]: r_star[i] for i in orden[:K]}   # PageRank de cada nodo del subgrafo

pos = nx.spring_layout(SG, seed=7, k=0.55)          # layout de fuerzas (semilla fija para reproducibilidad)
node_color = ['tab:blue' if values[v] == 0 else 'tab:red' for v in SG.nodes]
node_size = [3.2e5 * pr_map[v] for v in SG.nodes]   # tamaño proporcional al PageRank

fig, ax = plt.subplots(figsize=(11, 8.5))
nx.draw_networkx_edges(SG, pos, ax=ax, alpha=.16, arrowsize=6, width=.6)
nx.draw_networkx_nodes(SG, pos, ax=ax, node_color=node_color, node_size=node_size,
                       edgecolors='k', linewidths=.5)
# etiquetamos los 15 más importantes para no saturar la figura
nx.draw_networkx_labels(SG, pos, labels={nodes[i]: labels[nodes[i]] for i in orden[:15]},
                        font_size=7, ax=ax)
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([0],[0], marker='o', color='w', markerfacecolor='tab:blue', markersize=10, label='liberal (value=0)'),
                   Line2D([0],[0], marker='o', color='w', markerfacecolor='tab:red',  markersize=10, label='conservador (value=1)')],
          loc='upper left')
ax.set_title(f'Subgrafo inducido por los {K} nodos de mayor PageRank\n(tamaño ∝ PageRank, color = orientación política)')
ax.axis('off')
plt.tight_layout(); plt.savefig('figs/fig_subgrafo_top40.png', bbox_inches='tight'); plt.show()

# fracción de aristas dentro del mismo grupo
mismas = sum(1 for u, w in SG.edges if values[u] == values[w])
print(f"Aristas del subgrafo: {SG.number_of_edges()}  |  entre blogs de la MISMA orientación: "
      f"{mismas} ({100*mismas/SG.number_of_edges():.1f}%)")

El subgrafo se separa en dos comunidades, liberal (azul) y conservadora (roja), con pocos enlaces cruzados: el 81.9% de las 337 aristas conecta blogs de la misma orientación. La discusión de esta polarización está en el informe, en el séptimo punto.

### (c) ¿A qué grupo pertenecen los nodos de mayor PageRank?

In [ ]:
#P7(c): composición política del top y masa de PageRank por grupo
for K_ in (10, 20, 50):
    lib = sum(1 for i in orden[:K_] if values[nodes[i]] == 0)
    print(f"Top-{K_:2d}: {lib:2d} liberales  |  {K_-lib:2d} conservadores")

masa_lib = r_star[[i for i in range(n) if values[nodes[i]] == 0]].sum()
print(f"\nMasa total de PageRank: liberales = {masa_lib:.3f}  |  conservadores = {1-masa_lib:.3f}")
print(f"(Recordar la composición de la red: 588 liberales (48%) vs 636 conservadores (52%))")

El top-20 tiene mayoría conservadora (14 vs 6), pero la masa total de PageRank queda casi empatada (51.6% conservador vs 48.4% liberal), proporción muy parecida a la composición de la red. El análisis del balance de influencia entre comunidades se desarrolla en el informe, en el séptimo punto.

---
## P8 — Discusión, limitaciones y conclusiones

En el informe, en el octavo punto, se presenta el contraste con la hipótesis inicial, las limitaciones de PageRank en esta red, las preguntas nuevas que surgieron y la explicación para alguien sin conocimientos técnicos.

---
## Checklist de entrega (autoverificación)

| Ítem del checklist | Dónde está |
|---|---|
| Red descrita con fuente citada y tabla de estadísticas | P1 e informe (punto 2) |
| Pregunta e hipótesis documentadas | Informe (punto 3) |
| Análisis exploratorio con gráficos de distribución de grado | P3 |
| Construcción explicitada de $H$, $S$ y $G$ | P4 |
| Iteración de potencias propia + curva de convergencia | P5 |
| Tabla de los 20 nodos de mayor PageRank | P5(d) |
| PageRank vs. in-degree con gráfico y correlación | P6 |
| Visualización del subgrafo de los más importantes | P7(b) |
| Discusión final conectada con la hipótesis | Informe (punto 8) |
| Código reproducible en GitHub | https://github.com/Prodigy0101/Proyecto-2-Algebra-Lineal-Avanzada-y-Modelamiento |